# COMMUN

### Imports et configuration

In [1]:
import os
import yaml
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy.exc import SQLAlchemyError

### Charger variables d'environnement depuis .env

In [5]:
load_dotenv()

True

In [6]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
SQL_FILES_PATH = ROOT_DIR / "etl"
print(CONFIG_PATH)
print(SQL_FILES_PATH)

/Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/config.yml


### Définir le chemin racine du projet (quel que soit le dossier courant)

In [8]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

### Lecture du fichier config.yml

In [7]:
conf = load_config(CONFIG_PATH)
db_conf = conf['database']
sql_file_conf = conf['sqlfile']

NameError: name 'load_config' is not defined

# AHMED

In [9]:
def test_postgres_connection(db_config: dict, db_name: str | None = None) -> str:
    """
    Teste la connexion à une base PostgreSQL et retourne un message lisible.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut (souvent 'postgres')
        db_name (str | None): Nom de la base à tester. Si None, utilise db_config['db_default'].

    Returns:
        str: Message de succès avec la version PostgreSQL ou message d'erreur.
    """

    # Choix de la base : soit celle fournie, soit la base par défaut
    db_to_use = db_name or db_config["db_default"]

    # Création de l'URL de connexion PostgreSQL compatible SQLAlchemy
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_to_use}"
    )

    try:
        # Création de l'engine SQLAlchemy
        engine = create_engine(db_url)

        # Ouverture de la connexion
        with engine.connect() as conn:
            # Exécution d'une requête pour récupérer la version PostgreSQL
            version = conn.execute(text("SELECT version();")).scalar()

            # Message de succès
            message = (
                f"Connexion réussie à la base '{db_to_use}'."
                f"Version PostgreSQL : {version}"
            )
            return message

    except SQLAlchemyError as e:
        # Gestion des erreurs de connexion
        message = (
            f"Erreur de connexion à la base '{db_to_use}'."
            f"{e}"
        )
        return message

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()

# execution
test_postgres_connection(db_conf)

In [ ]:
def create_database(db_config: dict, db_name: str):
    """
    Vérifie si une base PostgreSQL existe et la crée si nécessaire.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut utilisée pour se connecter initialement
        db_name (str): Nom de la base à créer ou vérifier.

    Returns:
        None
    """

    # Construction de l'URL de connexion sur la base par défaut (souvent 'postgres')
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_config['db_default']}"
    )
    try:
        # Création de l'engine avec autocommit pour exécuter CREATE DATABASE
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        with engine.connect() as conn:
            # Vérifier si la base existe déjà
            result = conn.execute(
                text("SELECT 1 FROM pg_database WHERE datname = :dbname"),
                {"dbname": db_name}
            )
            exists = result.scalar()  # Récupère le premier résultat (1 si la base existe)

            if not exists:
                # Crée la base si elle n'existe pas
                conn.execute(text(f'CREATE DATABASE "{db_name}"'))
                print(f"Base '{db_name}' créée.")
            else:
                print(f"Base '{db_name}' existe déjà.")
    except SQLAlchemyError as e:
        # Gestion des erreurs SQLAlchemy
        print(f"Erreur lors de la vérification ou création de la base : {e}")

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()


# execution
create_database(db_conf, db_conf["db_accm"])

In [ ]:
def execute_sql_file(db_conf: dict, db_name: str, sql_file_path: str) -> str:

    db_to_use = db_name
    sql_file = Path(sql_file_path)
    if not sql_file.is_file():
        return f"Fichier SQL introuvable : {sql_file_path}"
    db_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_to_use}"
    )
    try:
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        # Lire le contenu du fichier SQL
        sql_commands = sql_file.read_text(encoding="utf-8")

        with engine.connect() as conn:
            conn.execute(text(sql_commands))
        return f"Fichier SQL '{sql_file_path}' exécuté avec succès sur la base '{db_to_use}'."
    except SQLAlchemyError as e:
        return f"Erreur lors de l'exécution du fichier SQL : {e}"
    finally:
        if 'engine' in locals():
            engine.dispose()
execute_sql_file(db_conf, db_conf["db_accm"], SQL_FILES_PATH/sql_file_conf["file_1"])

In [ ]:
def execute_sql_to_df(db_conf: dict, db_name: str, sql_file_path: str) -> pd.DataFrame:
    """
    Exécute une requête SQL depuis un fichier sur une base PostgreSQL et retourne le résultat sous forme de DataFrame.

    Args:
        db_conf (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut (optionnelle)
        db_name (str): Nom de la base de données sur laquelle exécuter la requête.
        sql_file_path (str): Chemin vers le fichier SQL contenant la ou les requêtes.

    Returns:
        pd.DataFrame: DataFrame contenant le résultat de la requête si des colonnes existent,
                      sinon un DataFrame avec un message d'information.
    """
    # Convertit le chemin du fichier SQL en objet Path pour faciliter les manipulations
    sql_file = Path(sql_file_path)

    # Vérifie que le fichier SQL existe, sinon retourne un DataFrame avec message d'erreur
    if not sql_file.is_file():
        print(f"Fichier SQL introuvable : {sql_file_path}")
        return pd.DataFrame({"info_message": [f"Fichier SQL introuvable : {sql_file_path}"]})

    # Construction de l'URL de connexion PostgreSQL compatible SQLAlchemy
    connection_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_name}"
    )
    try:
        # Création de l'objet engine SQLAlchemy avec autocommit pour exécuter DDL si nécessaire
        engine = create_engine(connection_url, isolation_level="AUTOCOMMIT")

        # Lecture du contenu du fichier SQL
        sql_text = sql_file.read_text(encoding="utf-8")

        # Ouverture d'une connexion à la base de données
        with engine.connect() as conn:
            # Exécution de la requête SQL
            result = conn.execute(text(sql_text))
            
            # Si la requête renvoie des lignes (ex: SELECT), créer un DataFrame
            if result.returns_rows:
                df = pd.DataFrame(result.fetchall(), columns=result.keys())
            else:
                # Si la requête ne renvoie rien (ex: CREATE TABLE), renvoyer un message
                df = pd.DataFrame({"info_message": ["Requête exécutée avec succès, pas de résultat à afficher."]})
            
            return df

    except SQLAlchemyError as e:
        # Capture des erreurs SQLAlchemy et retour d'un DataFrame contenant l'erreur
        return pd.DataFrame({"info_message": [f"Erreur lors de l'exécution : {e}"]})

    finally:
        # Libération des ressources de l'engine pour fermer proprement la connexion
        if 'engine' in locals():
            engine.dispose()


## Exemple

df = execute_sql_to_df(db_conf, db_conf["db_accm"], SQL_FILES_PATH/sql_file_conf["file_1"])
df.head()

# ROMAIN

In [13]:
API_CONFIG = {
    'base_url' : 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records',
    'limit_per_request' : 100,
    'max_records' : 1000,
    'timeout' : 30
}

print(f"API: {API_CONFIG['base_url'][:70]}...")

API: https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/acci...


In [9]:
def extraire_accidents_api(max_records=None):
    """
    Fonction pour extraire les accidents depuis l'API
    
    Paramètres:
        max_records (int): Le nombre maximum d'accidents à extraire. Si None, tous les accidents seront extraits.
    
    Return:
        list: Une liste de dictionnaires représentant les accidents extraîts.
    """

print("=" * 80)
print("EXTRACTION DES DONNÉES")
print("=" * 80)

all_records = []
offset = 0
limit = API_CONFIG['limit_per_request']

try:
    #première requête pour connaitre le total
    print("Récupération du nombre total...")
    response = requests.get(
        API_CONFIG['base_url'],
        params={'limit': 1},
        timeout=API_CONFIG['timeout']
    )
    response.raise_for_status()
    data = response.json()
    total_count = data.get('total_count', 0)
    
    print(f"Total disponible: {total_count:,} enregistrements")
    print(data)       
except requests.RequestException as e:
    print(f"\n✗ Erreur lors de l'extraction: {e}")
    raise

EXTRACTION DES DONNÉES
Récupération du nombre total...
Total disponible: 475,911 enregistrements
{'total_count': 475911, 'results': [{'num_acc': '201700009715', 'datetime': '2017-05-28T16:50:00+00:00', 'nom_com': None, 'an': '2017', 'mois': '05', 'jour': '28', 'hrmn': '18:50', 'lum': 'Plein jour', 'agg': 'En agglomération', 'int': '3', 'atm': 'Normale', 'col': 'Deux véhicules – par le coté', 'dep': '13', 'com': '055', 'insee': '13055', 'adr': '6 Av Alexandre  Ansaldi', 'lat': '4333582', 'long': '0539866', 'code_postal': None, 'num': '6', 'coordonnees': {'lon': 2.911777, 'lat': 42.686216}, 'pr': None, 'surf': 'normale', 'v1': None, 'circ': 'Bidirectionnelle', 'vosp': None, 'env1': '00', 'voie': '4', 'larrout': 120, 'v2': None, 'lartpc': 25, 'nbv': 4, 'catr': 'Route Départementale', 'pr1': None, 'plan': 'Partie rectiligne', 'prof': 'Plat', 'infra': None, 'situ': 'Sur chaussée', 'an_nais': ['1998', '1966'], 'sexe': ['Masculin', 'Masculin'], 'actp': ['Se déplaçant', 'Se déplaçant'], 'grav'

In [14]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""

import sys
from dataclasses import dataclass
from pathlib import Path
import requests



@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print(f"Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


Téléchargement depuis OpenDataSoft...


Fichier sauvegardé: /Users/LounesAbd/Etudes/Simplon_DataEng/Projects/1_trafic-road-accidents/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
Téléchargement terminé


In [8]:
# Import du script avec chemin absolu
import sys

# Chemin absolu vers le dossier scripts
scripts_dir = ROOT_DIR / 'etl'

# Ajouter au path Python
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Import
from sauvegarde_csv_api import download_accidents

# Téléchargement
stats = download_accidents(
    output=ROOT_DIR / 'data/accidents_corporels_millesime.csv',
    delimiter=';',
    show_progress=True,
    retries=3,
    #limit = 1000,
    #where ="an=2017 AND dep='60'"
)

print(f"✅ Téléchargement réussi !")
print(f"   - {stats.record_count:,} accidents")
print(f"   - {stats.column_count} colonnes")
print(f"   - {stats.size_mb:.2f} MB")


🚗 TÉLÉCHARGEMENT DATASET ACCIDENTS CORPORELS
📍 Source: OpenDataSoft
📅 Période: 2012-2019
📄 Fichier: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
🔤 Séparateur: ','

⚠️  Module 'tqdm' non installé - pas de barre de progression détaillée
   Installez-le avec: pip install tqdm

🌐 Connexion à OpenDataSoft...
📥 Téléchargement (taille inconnue)
   Téléchargé: 354,434.8 KB
✅ Fichier sauvegardé: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
⏱️  Temps de téléchargement: 321.3 secondes

📊 RÉSUMÉ DU TÉLÉCHARGEMENT
✅ Fichier valide
   Taille: 346.13 MB (362,941,194 bytes)
   Lignes: 475,912 (incluant l'en-tête)
   Records: 475,911 accidents
   Colonnes: 69

📋 Premières colonnes:
   1. num_acc
   2. datetime
   3. nom_com
   4. an
   5. mois
   6. jour
   7. hrmn
   8. lum
   9. agg
   10. int
   ... et 59 autres colonnes

✅ Téléchargement réussi !
   - 47

# LOUNES

In [10]:
# Importation des données source dans un DataFrame pandas

df_source = pd.read_csv(ROOT_DIR / 'data/accidents_corporels_millesime.csv', delimiter=',', low_memory=False)
df_source.head()

,num_acc,datetime,nom_com,an,mois,jour,hrmn,lum,agg,int,...,year_georef,com_name,dep_code,dep_name,epci_code,epci_name,reg_code,reg_name,com_arm_name,com_code
0,201900020750,2019-01-29T15:45:00+00:00,Corbeil-essonnes,2019,1,29,16:45,Plein jour,En agglomération,2,...,2019,Corbeil-Essonnes,91.0,Essonne,200059228.0,CA Grand Paris Sud Seine Essonne Sénart,11.0,Île-de-France,Corbeil-Essonnes,91174.0
1,201900020796,2019-10-07T17:30:00+00:00,Istres,2019,10,7,19:30,Nuit sans éclairage public,Hors agglomération,1,...,2019,Istres,13.0,Bouches-du-Rhône,200054807.0,Métropole d'Aix-Marseille-Provence,93.0,Provence-Alpes-Côte d'Azur,Istres,13047.0
2,201900020869,2019-10-13T13:46:00+00:00,Saint-laurent-du-pont,2019,10,13,15:46,Plein jour,En agglomération,1,...,2019,Saint-Laurent-du-Pont,38.0,Isère,200040111.0,CC Coeur de Chartreuse,84.0,Auvergne-Rhône-Alpes,Saint-Laurent-du-Pont,38412.0
3,201900021309,2019-04-17T14:30:00+00:00,Livry-gargan,2019,4,17,16:30,Plein jour,En agglomération,1,...,2019,Livry-Gargan,93.0,Seine-Saint-Denis,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Livry-Gargan,93046.0
4,201900018753,2019-12-25T17:10:00+00:00,Gennevilliers,2019,12,25,18:10,Nuit sans éclairage public,Hors agglomération,1,...,2019,Gennevilliers,92.0,Hauts-de-Seine,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Gennevilliers,92036.0


In [ ]:
# TEST / EXPLORATION
# Exploration de la donnée source csv

df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 69 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   num_acc       475911 non-null  int64  
 1   datetime      475911 non-null  object 
 2   nom_com       449206 non-null  object 
 3   an            475911 non-null  int64  
 4   mois          475911 non-null  int64  
 5   jour          475911 non-null  int64  
 6   hrmn          475911 non-null  object 
 7   lum           475911 non-null  object 
 8   agg           475911 non-null  object 
 9   int           475911 non-null  int64  
 10  atm           475861 non-null  object 
 11  col           475901 non-null  object 
 12  dep           475911 non-null  object 
 13  com           475911 non-null  object 
 14  insee         475296 non-null  float64
 15  adr           426655 non-null  object 
 16  lat           300785 non-null  object 
 17  long          300785 non-null  object 
 18  code

In [ ]:
"""
Préparation des données : nettoyage, transformation

Objectif : diviser le csv en 5 tables distinctes (accident, vehicule, usager, lieux, date)
Chaque table sera dans une dataframe pandas distincte

1ere étape : premier petit nettoyage et uniformisation des données
2eme étape : séparation en 5 dataframes
3ème étape : explosion des colonnes multi-valeurs pour les dataframes vehicule et usager
4eme étape : nettoyage spécifique à chaque dataframe
5ème étape : mapping des valeurs catégorielles pour chaque dataframe
6ème étape : tests de validation des données
7ème étape : export des dataframes nettoyés et validés dans postgres
"""
# FONCTION 1 (première transfo sur les noms de colonnes et datetime + création des 5 df source)
# Premier nettoyage et uniformisation des données
df_source.columns = df_source.columns.str.lower().str.strip()
df_source["datetime"] = pd.to_datetime(df_source["datetime"], errors="coerce")
df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 69 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   num_acc       475911 non-null  int64              
 1   datetime      475911 non-null  datetime64[ns, UTC]
 2   nom_com       449206 non-null  object             
 3   an            475911 non-null  int64              
 4   mois          475911 non-null  int64              
 5   jour          475911 non-null  int64              
 6   hrmn          475911 non-null  object             
 7   lum           475911 non-null  object             
 8   agg           475911 non-null  object             
 9   int           475911 non-null  int64              
 10  atm           475861 non-null  object             
 11  col           475901 non-null  object             
 12  dep           475911 non-null  object             
 13  com           475911 non-null  object       

In [ ]:
# FONCTION 1 - Séparation en 5 dataframes
# 1. Accidents
df_accidents = df_source[[
    'num_acc', 
    'lum', 
    'agg',
    'int',
    'atm',
    'adr',
    'col',
    'circ',
    'plan',
    'prof',
    'surf',
    'infra',
    'situ',
    'year_georef'
]].copy()

# EXPLORATION
# df_accidents.head()
df_accidents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   num_acc      475911 non-null  int64 
 1   lum          475911 non-null  object
 2   agg          475911 non-null  object
 3   int          475911 non-null  int64 
 4   atm          475861 non-null  object
 5   adr          426655 non-null  object
 6   col          475901 non-null  object
 7   circ         450504 non-null  object
 8   plan         441487 non-null  object
 9   prof         447014 non-null  object
 10  surf         459780 non-null  object
 11  infra        54500 non-null   object
 12  situ         449540 non-null  object
 13  year_georef  475911 non-null  int64 
dtypes: int64(3), object(11)
memory usage: 50.8+ MB


In [ ]:
# EXPLORATION
# Tests de validation dataframe df_accidents

# Vérification des doublons sur 'num_acc
duplicates_accidents = df_accidents.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_accidents sur 'num_acc': {duplicates_accidents} \n")

# Vérification des valeurs uniques sur les catégories
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    unique_values = df_accidents[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    nan_count = df_accidents[col].isna().sum()
    neg_one_count = (df_accidents[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


Doublons dans df_accidents sur 'num_acc': 0 


 Valeurs uniques dans 'lum': ['Plein jour' 'Nuit sans éclairage public'
 'Nuit avec éclairage public non allumé'
 'Nuit avec éclairage public allumé' 'Crépuscule ou aube']

 Valeurs uniques dans 'agg': ['En agglomération' 'Hors agglomération']

 Valeurs uniques dans 'int': [2 1 9 6 3 7 4 5 0 8]

 Valeurs uniques dans 'atm': ['Normale' 'Temps couvert' 'Temps éblouissant' 'Pluie légère'
 'Pluie forte' 'Autre' 'Vent fort - tempête' 'Brouillard - fumée'
 'Neige - grêle' nan '-1']

 Valeurs uniques dans 'col': ['Deux véhicules – par le coté' 'Sans collision'
 'Deux véhicules - frontale' 'Deux véhicules – par l’arrière'
 'Autre collision' 'Trois véhicules et plus - collisions multiples'
 'Trois véhicules et plus – en chaîne' nan '-1']

 Valeurs uniques dans 'circ': ['Bidirectionnelle' '-1' 'A sens unique' 'A chaussées séparées' nan
 'Avec voies d’affectation variable']

 Valeurs uniques dans 'plan': ['Partie rectiligne' nan 'En courbe à droite' 

In [ ]:
# FONCTION 2 - TRANSFORMATION DF ACCIDENTS
# Transformation des valeurs '-1' en NaN pour les colonnes catégorielles
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    df_accidents[col] = df_accidents[col].replace('-1', np.nan)

In [ ]:
# FONCTION 2 - TRANSFORMATION DF ACCIDENTS
# Nettoyer "infra" et "situ" car ont des numéros écris en type object et des catégories mélangées. Transformer les numéros en NaN
df_accidents['infra'] = df_accidents['infra'].apply(lambda x: np.nan if str(x).isdigit() else x)
df_accidents['situ'] = df_accidents['situ'].apply(lambda x: np.nan if str(x).isdigit() else x)

In [ ]:
# La dataframe df_accidents est prête pour export. Mapping catégories à réaliser.



In [ ]:
# FONCTION 1 - Séparation en 5 dataframes
# 2. Lieux
df_lieux = df_source[[
    'com_code',
    'com_name',
    'dep_code',
    'dep_name',
    'reg_code', 
    'reg_name', 
    'epci_code', 
    'epci_name',
    'lat', 
    'long', 
    'catr', 
    'v1', 
    'voie', 
    'v2', 
    'nbv', 
    'vosp', 
    'pr', 
    'pr1', 
    'lartpc', 
    'larrout', 
    'num_acc'
]].copy()

# EXPLORATION
# df_lieux.head()
df_lieux.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 21 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   com_code   475296 non-null  float64
 1   com_name   464648 non-null  object 
 2   dep_code   464648 non-null  float64
 3   dep_name   464648 non-null  object 
 4   reg_code   464648 non-null  float64
 5   reg_name   464648 non-null  object 
 6   epci_code  426190 non-null  float64
 7   epci_name  426190 non-null  object 
 8   lat        300785 non-null  object 
 9   long       300785 non-null  object 
 10  catr       475911 non-null  object 
 11  v1         50267 non-null   float64
 12  voie       410712 non-null  object 
 13  v2         19498 non-null   object 
 14  nbv        474103 non-null  float64
 15  vosp       32080 non-null   object 
 16  pr         259060 non-null  object 
 17  pr1        250140 non-null  float64
 18  lartpc     362782 non-null  float64
 19  larrout    364392 non-n

In [ ]:
# EXPLORATION
# Vérification des doublons sur 'num_acc
duplicates_lieux = df_lieux.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_lieux sur 'num_acc': {duplicates_lieux} \n")

# Vérification des catégories
# df_lieux['catr'].value_counts(dropna=False)

# Notes nettoyage : 
# drop 'voie', drop 'V1', drop 'V2', drop 'pr', drop 'pr1', drop 'lartpc', drop 'larrout, drop 'epci_code', drop 'epci_name', drop 'lat', drop 'long
# numéros à mettre en NaN dans 'catr'
# nbv > 6 ou nbv = -1 ou nbv = 0 --> NaN (explication : deux fois 3 voies sur autoroute est le max selon nous soit 6 voies)
# vosp = -1 --> NaN

# FONCTION 3 - TRANSFORMATION DF LIEUX
# Drop des colonnes inutiles
df_lieux = df_lieux.drop(columns=['voie', 'v1', 'v2', 'pr', 'pr1', 'lartpc', 'larrout', 'epci_code', 'epci_name', 'lat', 'long'])


Doublons dans df_lieux sur 'num_acc': 0 



In [ ]:
# FONCTION 3 - TRANSFORMATION DF LIEUX
# Nettoyer "catr"
df_lieux['catr'] = df_lieux['catr'].apply(lambda x: 'autre' if str(x).isdigit() else x)

df_lieux['catr'].value_counts(dropna=False)

catr
Voie Communale                                            230977
Route Départementale                                      161026
Autoroute                                                  41951
Route Nationale                                            30360
autre                                                       7492
Parc de stationnement ouvert à la circulation publique      3334
Hors réseau public                                           771
Name: count, dtype: int64

In [ ]:
# FONCTION 3 - TRANSFORMATION DF LIEUX
# Nettoyer 'nbv'
# nbv > 6 ou nbv = -1 ou nbv = 0 --> NaN (explication : deux fois 3 voies sur autoroute est le max selon nous soit 6 voies). values are float.
df_lieux['nbv'] = df_lieux['nbv'].apply(lambda x: np.nan if (x > 6.0 or x == 0.0) or x == -1.0 else x)

# transformer le dtype en int
df_lieux['nbv'] = df_lieux['nbv'].astype('Int64')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   com_code  475296 non-null  float64
 1   com_name  464648 non-null  object 
 2   dep_code  464648 non-null  float64
 3   dep_name  464648 non-null  object 
 4   reg_code  464648 non-null  float64
 5   reg_name  464648 non-null  object 
 6   catr      475911 non-null  object 
 7   nbv       424073 non-null  Int64  
 8   vosp      32080 non-null   object 
 9   num_acc   475911 non-null  int64  
dtypes: Int64(1), float64(3), int64(1), object(5)
memory usage: 36.8+ MB


In [ ]:
# FONCTION 3 - TRANSFORMATION DF LIEUX
# Nettoyer 'vosp'
# vosp = -1 --> NaN
df_lieux['vosp'] = df_lieux['vosp'].apply(lambda x: np.nan if x == '-1' else x)
df_lieux['vosp'].value_counts(dropna=False)

vosp
NaN                444500
Voie réservée       13686
Piste cyclable      11258
Banque cyclable      6467
Name: count, dtype: int64

In [ ]:
# EXPLORATION
# Last check df_lieux
df_lieux.info()

# df_lieux est prêt pour export dans postgres

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   com_code  475296 non-null  float64
 1   com_name  464648 non-null  object 
 2   dep_code  464648 non-null  float64
 3   dep_name  464648 non-null  object 
 4   reg_code  464648 non-null  float64
 5   reg_name  464648 non-null  object 
 6   catr      475911 non-null  object 
 7   nbv       424073 non-null  Int64  
 8   vosp      31411 non-null   object 
 9   num_acc   475911 non-null  int64  
dtypes: Int64(1), float64(3), int64(1), object(5)
memory usage: 36.8+ MB


In [ ]:
# FONCTION 1 - Séparation en 5 dataframes

# 3. Date_accident
df_date_accident = df_source[[
    'datetime', 
    'an', 
    'mois', 
    'jour', 
    'hrmn', 
    'num_acc'
]].copy()

# EXPLORATION
# df_date_accident.head()
df_date_accident.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB


In [ ]:
# EXPLORATION
# Vérification des colonnes 'an', 'mois', 'jour' pour éviter des aberrations
df_date_accident[['an', 'mois', 'jour']].describe()

# FONCTION 4 - TRANSFORMATION DF DATE_ACCIDENT
# Vérification et transformation du dtype de la colonne 'hrmn' (exemple : dtype object "16:45") en datetime object "HH:MM"
df_date_accident['hrmn'] = pd.to_datetime(df_date_accident['hrmn'], format='%H:%M', errors='coerce').dt.time

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB


In [ ]:
# EXPLORATION
# Last check for df_date_accident
df_date_accident.info()

# df_date_accident est prêt pour export dans postgres

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB


In [ ]:
# FONCTION 1 - Séparation en 5 dataframes
 
# 4. Vehicules
df_vehicules = df_source[[
    'num_veh', 
    'catv', 
    'choc', 
    'senc', 
    'obs', 
    'obsm', 
    'occutc', 
    'manv', 
    'num_acc'
]].copy()

# EXPLORATION
df_vehicules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 9 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   num_veh  475911 non-null  object
 1   catv     475911 non-null  object
 2   choc     446407 non-null  object
 3   senc     209666 non-null  object
 4   obs      101364 non-null  object
 5   obsm     369261 non-null  object
 6   occutc   6015 non-null    object
 7   manv     441491 non-null  object
 8   num_acc  475911 non-null  int64 
dtypes: int64(1), object(8)
memory usage: 32.7+ MB


In [ ]:
# EXPLORATION
df_vehicules.head(20)

,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,"B01,A01","Cyclomoteur <50cm3,VL seul",Avant,"3,3",NaN,Véhicule,NaN,Traversant la chaussée,201900020750
1,A01,VU seul 1,NaN,3,NaN,NaN,NaN,Même sens,201900020796
2,"B01,A01","VL seul,VL seul","Avant gauche,Avant",PK ou PR ou numéro d’adresse postale croissant...,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Déporté A gauche",201900020869
3,"B01,A01","43,VL seul",Avant,PK ou PR ou numéro d’adresse postale croissant...,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Manœuvre de stati...",201900021309
4,"C01,B01,A01","VL seul,VL seul,VL seul","Arrière,Arrière,Avant",PK ou PR ou numéro d’adresse postale décroissa...,NaN,"Véhicule,Véhicule,Véhicule",NaN,"Même sens,Même sens,Même sens",201900018753
5,"A01,B01","VL seul,VL seul","Avant,Côté gauche","3,3",NaN,"Véhicule,Véhicule",NaN,"Tournant A gauche,Sans changement de direction",201900037513
6,"C01,B01,A01","VL seul,VL seul,VL seul","Côté gauche,Côté gauche,Avant",NaN,NaN,"Véhicule,Véhicule,Véhicule",NaN,"Sans changement de direction,Sans changement d...",201400048899
7,A01,Motocyclette > 125 cm3,Avant,NaN,NaN,Autre,NaN,Entre 2 files,201400049030
8,A01,Autre véhicule,Avant,PK ou PR ou numéro d’adresse postale croissant,NaN,Piéton,NaN,Sans changement de direction,201700041615
9,"B01,A01","Engin spécial,VL seul","Avant,Arrière",PK ou PR ou numéro d’adresse postale croissant...,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Arrêté (hors stat...",201700042289


In [ ]:
# FONCTION 5 - EXPLOSION DF VEHICULES ET DF USAGERS
def explode_df(df, cols_to_explode, id_col_name):
    """
    Explodes a DataFrame where some columns contain comma-separated values.
    Handles list alignment, padding, and adds a sequential unique ID column.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    cols_to_explode : list
        List of column names that contain comma-separated or list values.
    id_col_name : str
        Name of the ID column to create (e.g. 'vehicule_id', 'usager_id').
    
    Returns
    -------
    pd.DataFrame
        Exploded DataFrame with a unique sequential ID as the first column.
    """
    df = df.copy()

    # 1️⃣ Ensure all values are lists (split by comma if needed)
    for col in cols_to_explode:
        df[col] = df[col].astype(str).apply(lambda x: x.split(',') if ',' in x else [x])

    # 2️⃣ Compute length consistency per row
    list_lengths_df = df[cols_to_explode].map(lambda x: len(x) if isinstance(x, list) else 1)
    df['nunique_lengths'] = list_lengths_df.nunique(axis=1)
    df['max_length'] = list_lengths_df.max(axis=1)

    # 3️⃣ Detect misaligned rows
    misaligned_rows = df[df['nunique_lengths'] > 1]
    if len(misaligned_rows) > 0:
        print(f"⚠️ {len(misaligned_rows)} rows with misaligned list lengths detected — they will be padded.")

    # 4️⃣ Pad lists to same length
    def pad_lists(row):
        max_len = row['max_length']
        for col in cols_to_explode:
            vals = row[col] if isinstance(row[col], list) else [row[col]]
            row[col] = (vals + [None] * (max_len - len(vals)))[:max_len]
        return row

    df = df.apply(pad_lists, axis=1)

    # 5️⃣ Explode all relevant columns together
    df_exploded = df.explode(cols_to_explode, ignore_index=True)
    print(f"✓ Data exploded successfully: {len(df_exploded):,} rows.")

    # 6️⃣ Add sequential unique ID
    df_exploded = df_exploded.reset_index(drop=True)
    df_exploded[id_col_name] = df_exploded.index + 1

    # 7️⃣ Reorder columns to put ID first
    cols = [id_col_name] + [c for c in df_exploded.columns if c != id_col_name]
    df_exploded = df_exploded[cols]

    return df_exploded


In [ ]:
# EXPLORATION
# Transformer les colonnes qui contiennent plusieurs valeurs séparées par des virgules en listes
veh_cols = ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']
for col in veh_cols:
    df_vehicules[col] = df_vehicules[col].astype(str).apply(lambda x: x.split(',') if ',' in x else [x])

df_vehicules.head(20)

,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,"[B01, A01]","[Cyclomoteur <50cm3, VL seul]",[Avant],"[3, 3]",[nan],[Véhicule],[nan],[Traversant la chaussée],201900020750
1,[A01],[VU seul 1],[nan],[3],[nan],[nan],[nan],[Même sens],201900020796
2,"[B01, A01]","[VL seul, VL seul]","[Avant gauche, Avant]",[PK ou PR ou numéro d’adresse postale croissan...,[nan],"[Véhicule, Véhicule]",[nan],"[Sans changement de direction, Déporté A gauche]",201900020869
3,"[B01, A01]","[43, VL seul]",[Avant],[PK ou PR ou numéro d’adresse postale croissan...,[nan],"[Véhicule, Véhicule]",[nan],"[Sans changement de direction, Manœuvre de sta...",201900021309
4,"[C01, B01, A01]","[VL seul, VL seul, VL seul]","[Arrière, Arrière, Avant]",[PK ou PR ou numéro d’adresse postale décroiss...,[nan],"[Véhicule, Véhicule, Véhicule]",[nan],"[Même sens, Même sens, Même sens]",201900018753
5,"[A01, B01]","[VL seul, VL seul]","[Avant, Côté gauche]","[3, 3]",[nan],"[Véhicule, Véhicule]",[nan],"[Tournant A gauche, Sans changement de direction]",201900037513
6,"[C01, B01, A01]","[VL seul, VL seul, VL seul]","[Côté gauche, Côté gauche, Avant]",[nan],[nan],"[Véhicule, Véhicule, Véhicule]",[nan],"[Sans changement de direction, Sans changement...",201400048899
7,[A01],[Motocyclette > 125 cm3],[Avant],[nan],[nan],[Autre],[nan],[Entre 2 files],201400049030
8,[A01],[Autre véhicule],[Avant],[PK ou PR ou numéro d’adresse postale croissant],[nan],[Piéton],[nan],[Sans changement de direction],201700041615
9,"[B01, A01]","[Engin spécial, VL seul]","[Avant, Arrière]",[PK ou PR ou numéro d’adresse postale croissan...,[nan],"[Véhicule, Véhicule]",[nan],"[Sans changement de direction, Arrêté (hors st...",201700042289


In [ ]:
# EXPLORATION

# 1️⃣ Compute length of each list cell for the vehicule columns
list_lengths_df = df_vehicules[veh_cols].map(
    lambda x: len(x) if isinstance(x, list) else 1
)

# 2️⃣ Compute how many distinct lengths there are per row
df_vehicules['nunique_lengths'] = list_lengths_df.nunique(axis=1)

# 3️⃣ (Optional) Store the max length — useful for padding later
df_vehicules['max_length'] = list_lengths_df.max(axis=1)

# 4️⃣ Identify misaligned rows (drifting)
misaligned_rows = df_vehicules[df_vehicules['nunique_lengths'] > 1]

print(f"⚠️ {len(misaligned_rows)} rows with misaligned list lengths detected.")

⚠️ 290537 rows with misaligned list lengths detected.


In [ ]:
# EXPLORATION

def pad_lists(row):
    max_len = row['max_length']
    for col in veh_cols:
        vals = row[col] if isinstance(row[col], list) else [row[col]]
        row[col] = (vals + [None] * (max_len - len(vals)))[:max_len]
    return row

df_vehicules = df_vehicules.apply(pad_lists, axis=1)

In [ ]:
# EXPLORATION

# Retirer les colonnes inutiles 'nunique_lengths', 'max_length' et 'senc'
df_vehicules = df_vehicules.drop(columns=['nunique_lengths', 'max_length', 'senc'])

df_vehicules.head(10)

,num_veh,catv,choc,obs,obsm,occutc,manv,num_acc
0,"B01,A01","Cyclomoteur <50cm3,VL seul",Avant,NaN,Véhicule,NaN,Traversant la chaussée,201900020750
1,A01,VU seul 1,NaN,NaN,NaN,NaN,Même sens,201900020796
2,"B01,A01","VL seul,VL seul","Avant gauche,Avant",NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Déporté A gauche",201900020869
3,"B01,A01","43,VL seul",Avant,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Manœuvre de stati...",201900021309
4,"C01,B01,A01","VL seul,VL seul,VL seul","Arrière,Arrière,Avant",NaN,"Véhicule,Véhicule,Véhicule",NaN,"Même sens,Même sens,Même sens",201900018753
5,"A01,B01","VL seul,VL seul","Avant,Côté gauche",NaN,"Véhicule,Véhicule",NaN,"Tournant A gauche,Sans changement de direction",201900037513
6,"C01,B01,A01","VL seul,VL seul,VL seul","Côté gauche,Côté gauche,Avant",NaN,"Véhicule,Véhicule,Véhicule",NaN,"Sans changement de direction,Sans changement d...",201400048899
7,A01,Motocyclette > 125 cm3,Avant,NaN,Autre,NaN,Entre 2 files,201400049030
8,A01,Autre véhicule,Avant,NaN,Piéton,NaN,Sans changement de direction,201700041615
9,"B01,A01","Engin spécial,VL seul","Avant,Arrière",NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Arrêté (hors stat...",201700042289


In [ ]:
# EXPLORATION

# Make a copy to avoid modifying the original
df_veh_exploded = df_vehicules.copy()

# 🚀 Explode all list-type columns together (they all have equal-length lists now)
df_veh_exploded = df_veh_exploded.explode(veh_cols, ignore_index=True)

print(f"✓ Data exploded successfully: {len(df_veh_exploded):,} rows.")

# Ajout d'un identifiant unique pour chaque véhicule
df_veh_exploded = df_veh_exploded.reset_index(drop=True)
df_veh_exploded["vehicule_id"] = df_veh_exploded.index + 1

# Réorganisation des colonnes pour mettre l'id en premier
cols = ["vehicule_id"] + [c for c in df_veh_exploded.columns if c != "vehicule_id"]
df_veh_exploded = df_veh_exploded[cols]

df_veh_exploded.head(10)

✓ Data exploded successfully: 811,335 rows.


,vehicule_id,num_veh,catv,choc,obs,obsm,occutc,manv,num_acc
0,1,B01,Cyclomoteur <50cm3,Avant,nan,Véhicule,nan,Traversant la chaussée,201900020750
1,2,A01,VL seul,None,None,None,None,None,201900020750
2,3,A01,VU seul 1,nan,nan,nan,nan,Même sens,201900020796
3,4,B01,VL seul,Avant gauche,nan,Véhicule,nan,Sans changement de direction,201900020869
4,5,A01,VL seul,Avant,None,Véhicule,None,Déporté A gauche,201900020869
5,6,B01,43,Avant,nan,Véhicule,nan,Sans changement de direction,201900021309
6,7,A01,VL seul,None,None,Véhicule,None,Manœuvre de stationnement,201900021309
7,8,C01,VL seul,Arrière,nan,Véhicule,nan,Même sens,201900018753
8,9,B01,VL seul,Arrière,None,Véhicule,None,Même sens,201900018753
9,10,A01,VL seul,Avant,None,Véhicule,None,Même sens,201900018753


In [ ]:
# EXPLORATION

# Test de la fonction explode_df pour df_vehicules
df_veh_exploded = explode_df(df_vehicules, ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv'], 'vehicule_id')
df_veh_exploded.head()

⚠️ 290537 rows with misaligned list lengths detected — they will be padded.
✓ Data exploded successfully: 811,335 rows.


,vehicule_id,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc,nunique_lengths,max_length
0,1,B01,Cyclomoteur <50cm3,Avant,3,nan,Véhicule,nan,Traversant la chaussée,201900020750,2,2
1,2,A01,VL seul,None,3,None,None,None,None,201900020750,2,2
2,3,A01,VU seul 1,nan,3,nan,nan,nan,Même sens,201900020796,1,1
3,4,B01,VL seul,Avant gauche,PK ou PR ou numéro d’adresse postale croissant,nan,Véhicule,nan,Sans changement de direction,201900020869,2,2
4,5,A01,VL seul,Avant,PK ou PR ou numéro d’adresse postale décroissant,None,Véhicule,None,Déporté A gauche,201900020869,2,2


In [ ]:
# EXPLORATION

# Vérification des données après explosion
# df_veh_exploded.info()

# Each num_acc now has one row per vehicle
check = df_veh_exploded.groupby("num_acc")["num_veh"].nunique().describe()
print(check)

count    475911.000000
mean          1.704693
std           0.683776
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          35.000000
Name: num_veh, dtype: float64


In [ ]:
# EXPLORATION

# Vérification des données après explosion

# Vérification des valeurs uniques sur les catégories
for col in ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    unique_values = df_veh_exploded[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    nan_count = df_veh_exploded[col].isna().sum()
    neg_one_count = (df_veh_exploded[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


 Valeurs uniques dans 'num_veh': ['B01' 'A01' 'C01' 'B02' 'Z01' 'C03' 'D01' 'F02' 'E01' 'Y01' 'F01' 'X01'
 'D04' 'C04' 'B03' 'E05' 'C02' 'A02' 'F06' 'G07' 'H08' 'D02' 'E02' 'I01'
 'H01' 'G01' 'Z02' 'G02' 'N01' 'R01' 'H02' 'T01' 'O01' 'J01' 'K01' 'L01'
 'M01' 'I09' 'TB01' 'AA01' 'A03' 'I02' 'N03' 'L03' 'M03' 'J02' 'O03' 'K03'
 'U01' 'P01' 'BB01' 'CB01' 'V01' 'ZZ01' 'BA01' 'G06' 'H07' 'D03' 'E04'
 'F05' 'I08' 'VB01' 'A04' 'M13' 'J10' 'K11' 'L12' 'O15' 'N14' 'W01' 'A05'
 'MA01' 'C08' 'S01' 'DA01' 'Z03' 'Z04' 'VF01' 'L02' 'K02' 'P16' 'W23'
 'S19' 'Y25' 'X24' 'U21' 'Q17' 'R18' 'V22' 'T20' 'Z26' 'B28' 'A27' 'RC01'
 'G33' 'I35' 'M39' 'D30' 'N40' 'E31' 'C29' 'K37' 'Q01' 'AB01' 'PB01' 'B05'
 'X02' 'E03' '[01' 'RA01' '\\01' 'MB01' 'LB01' 'C09' 'TC01' 'BC01' 'FB01'
 'GB01' 'ZB01']

 Valeurs uniques dans 'catv': ['Cyclomoteur <50cm3' 'VL seul' 'VU seul 1' '43' 'Motocyclette > 125 cm3'
 'Autre véhicule' 'Engin spécial' 'Scooter < 50 cm3' 'Tracteur agricole'
 'Motocyclette > 50 cm3 et <= 125 cm3' '

In [ ]:
# EXPLORATION

# Check des dtypes des colonnes
df_veh_exploded.dtypes

vehicule_id         int64
num_veh            object
catv               object
choc               object
senc               object
obs                object
obsm               object
occutc             object
manv               object
num_acc             int64
nunique_lengths     int64
max_length          int64
dtype: object

In [ ]:
# FONCTION 6 - TRANSFORMATION DF VEHICULES

# Check et nettoyage des colonnes catégorielles

# catv : retirer les parenthèses après "Voiturette"
# catv : 0 --> "Autre véhicule"
# catv : [50, 43, 42, 41, 60, 80] --> "Autre véhicule"
# catv : "Quad léger" & "Quad lourd" --> retirer les parenthèses
def clean_catv(value):
    if pd.isna(value):
        return value
    if value == '0':
        return 'Autre véhicule'
    if value in ['50', '43', '42', '41', '60', '80']:
        return 'Autre véhicule'
    if 'Voiturette (Quadricycle à moteur carrossé) (anciennement "voiturette ou tricycle à moteur")' in value:
        return 'Voiturette'
    if 'Quad léger <= 50 cm3 (Quadricycle à moteur non carrossé)' in value:
        return 'Quad léger <= 50 cm3'
    if 'Quad lourd > 50 cm3 (Quadricycle à moteur non carrossé)' in value:
        return 'Quad lourd > 50 cm3'
    return value

df_veh_exploded['catv'] = df_veh_exploded['catv'].apply(clean_catv)

# EXPLORATION
df_veh_exploded['catv'].value_counts(dropna=False)

catv
VL seul                                502874
Motocyclette > 125 cm3                  67462
VU seul 1                               46223
Bicyclette                              37757
Scooter < 50 cm3                        30904
Cyclomoteur <50cm3                      30456
Motocyclette > 50 cm3 et <= 125 cm3     19094
Scooter > 50 cm3 et <= 125 cm3          17870
Scooter > 125 cm3                       12258
PL seul > 7                              7690
PL > 3                                   7607
Autobus                                  6070
Tracteur routier + semi-remorque         5779
Autre véhicule                           5103
Voiturette                               3871
PL seul 3                                3120
Tracteur agricole                        1685
Autocar                                  1612
Quad lourd > 50 cm3                      1120
Tramway                                  1108
Engin spécial                             891
Tracteur routier seul        

In [ ]:
# FONCTION 6 - TRANSFORMATION DF VEHICULES

# clean : choc, obs, obsm, occutc, manv
# common rules to apply :
# '-1' --> NaN
# 'nan' or None --> np.nan
def clean_generic(value):
    if pd.isna(value) or value in ['nan', 'None']:
        return np.nan
    if value == '-1':
        return np.nan
    return value

for col in ['choc','obs', 'obsm', 'occutc', 'manv']:
    df_veh_exploded[col] = df_veh_exploded[col].apply(clean_generic)

# EXPLORATION
for col in ['choc', 'obs', 'obsm', 'occutc', 'manv']:
    print(f"\nColumn: {col}")
    print(df_veh_exploded[col].value_counts(dropna=False).head())


Column: choc
choc
Avant           291698
Avant gauche    116177
Avant droit      94729
Arrière          80746
Côté gauche      58228
Name: count, dtype: int64

Column: obs
obs
NaN                          705045
Véhicule en stationnement     17278
Fossé                         13196
Arbre                         11537
Glissière béton               10002
Name: count, dtype: int64

Column: obsm
obsm
Véhicule          551952
NaN               158246
Piéton             88109
Autre               9431
Animal sauvage      2209
Name: count, dtype: int64

Column: occutc
occutc
NaN    805247
1        3598
2         643
3         369
5         242
Name: count, dtype: int64

Column: manv
manv
Sans changement de direction    337837
Même sens                        99282
NaN                              71102
Tournant A gauche                67125
Déporté A gauche                 35367
Name: count, dtype: int64


In [ ]:
# FONCTION 6 - TRANSFORMATION DF VEHICULES

# change 'occutc' dtype to Int64
# change numbers to string for 'obs' and 'manv', keep dtype object
for cols in ['obs', 'manv']:
    df_veh_exploded[cols] = df_veh_exploded[cols].apply(lambda x: np.nan if str(x).isdigit() else x)

df_veh_exploded['occutc'] = df_veh_exploded['occutc'].astype('Int64')

df_veh_exploded = df_veh_exploded.drop(columns=['nunique_lengths', 'max_length'])

df_veh_exploded.head()

,vehicule_id,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,1,B01,Cyclomoteur <50cm3,Avant,3,NaN,Véhicule,<NA>,Traversant la chaussée,201900020750
1,2,A01,VL seul,NaN,3,NaN,NaN,<NA>,NaN,201900020750
2,3,A01,VU seul 1,NaN,3,NaN,NaN,<NA>,Même sens,201900020796
3,4,B01,VL seul,Avant gauche,PK ou PR ou numéro d’adresse postale croissant,NaN,Véhicule,<NA>,Sans changement de direction,201900020869
4,5,A01,VL seul,Avant,PK ou PR ou numéro d’adresse postale décroissant,NaN,Véhicule,<NA>,Déporté A gauche,201900020869


In [ ]:
# Dataframe véhicules prêt à l'export


In [ ]:
# FONCTION 1 - Séparation en 5 dataframes

# 5. Usagers
df_usagers = df_source[[
    'sexe', 
    'grav', 
    'trajet', 
    'secu', 
    'secu_utl', 
    'catu', 
    'place', 
    'locp', 
    'actp', 
    'etatp', 
    'num_acc'
]].copy()

# EXPLORATION
df_usagers.head()
#df_usagers.info()

,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,"Masculin,Masculin,Masculin","Blessé,Blessé,Indemne","Promenade – loisirs,Promenade – loisirs",NaN,NaN,"Conducteur,Passager,Conducteur","1,2,1",-1,"-1,Se déplaçant,Se déplaçant","-1,-1,-1",201900020750
1,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,-1,-1,-1,201900020796
2,"Masculin,Masculin,Masculin,Masculin,Masculin","Indemne,Indemne,Blessé,Indemne,Indemne","Promenade – loisirs,Promenade – loisirs,Promen...",NaN,NaN,"Conducteur,Passager,Conducteur,Passager,Passager","1,9,1,7,2",NaN,"Se déplaçant,Se déplaçant,Se déplaçant,Se dépl...","-1,-1,-1,-1,-1",201900020869
3,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,-1,-1,-1,201900021309
4,"Féminin,Masculin,Masculin,Féminin,Masculin,Fém...","Indemne,Blessé,Indemne,Indemne,Indemne,Indemne","Promenade – loisirs,Promenade – loisirs,Promen...",NaN,NaN,"Conducteur,Passager,Passager,Passager,Passager...","1,2,2,3,4,1","-1,-1,-1,-1,-1,-1","Se déplaçant,Se déplaçant,Se déplaçant,Se dépl...","-1,-1,-1,-1,-1,-1",201900018753


In [ ]:
# EXPLORATION

df_usagers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 11 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   sexe      475911 non-null  object
 1   grav      475911 non-null  object
 2   trajet    402358 non-null  object
 3   secu      413759 non-null  object
 4   secu_utl  413759 non-null  object
 5   catu      475911 non-null  object
 6   place     469630 non-null  object
 7   locp      101842 non-null  object
 8   actp      454084 non-null  object
 9   etatp     132740 non-null  object
 10  num_acc   475911 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 39.9+ MB


In [ ]:
# EXPLORATION

# Test fonction explode_df pour df_usagers

# Définir les colonnes à exploser
cols_to_explode = ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']

df_usager_exploded = explode_df(df_usagers, cols_to_explode, 'usager_id')
df_usager_exploded.head()

⚠️ 385795 rows with misaligned list lengths detected — they will be padded.
✓ Data exploded successfully: 1,061,254 rows.


,usager_id,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc,nunique_lengths,max_length
0,1,Masculin,Blessé,Promenade – loisirs,nan,nan,Conducteur,1,-1,-1,-1,201900020750,3,3
1,2,Masculin,Blessé,Promenade – loisirs,None,None,Passager,2,None,Se déplaçant,-1,201900020750,3,3
2,3,Masculin,Indemne,None,None,None,Conducteur,1,None,Se déplaçant,-1,201900020750,3,3
3,4,Masculin,Blessé,nan,nan,nan,Conducteur,1,-1,-1,-1,201900020796,1,1
4,5,Masculin,Indemne,Promenade – loisirs,nan,nan,Conducteur,1,nan,Se déplaçant,-1,201900020869,2,5


In [ ]:
# FONCTION 7 - TRANSFORMATION DF USAGERS

# Drop des colonnes inutiles 'nunique_lengths', 'max_length'
df_usager_exploded = df_usager_exploded.drop(columns=['nunique_lengths', 'max_length'])


# EXPLORATION

# Vérification des données après explosion
# Vérification des valeurs uniques sur les catégories
for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    unique_values = df_usager_exploded[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    nan_count = df_usager_exploded[col].isna().sum()
    neg_one_count = (df_usager_exploded[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


 Valeurs uniques dans 'sexe': ['Masculin' 'Féminin']

 Valeurs uniques dans 'grav': ['Blessé' 'Indemne' 'Tué']

 Valeurs uniques dans 'trajet': ['Promenade – loisirs' None 'nan' 'Utilisation professionnelle'
 'Domicile – travail' 'Courses – achats' 'Autre' 'Domicile – école' '-1']

 Valeurs uniques dans 'secu': ['nan' None 'Ceinture' 'Casque' 'Autre' 'Equipement réfléchissant'
 'Dispositif enfants']

 Valeurs uniques dans 'secu_utl': ['nan' None 'Oui' 'Non déterminable' 'Non']

 Valeurs uniques dans 'catu': ['Conducteur' 'Passager' 'Piéton' 'Piéton en roller ou en trottinette']

 Valeurs uniques dans 'place': ['1' '2' '9' '7' '3' '4' None 'nan' '5' '10' '8' '6']

 Valeurs uniques dans 'locp': ['-1' None 'nan' 'Sur contre allée'
 'Sur passage piéton - Avec signalisation lumineuse'
 'Sur passage piéton - Sans signalisation lumineuse'
 'Sur chaussée - A – 50 m du passage piéton'
 'Sur chaussée - A + 50 m du passage piéton' 'Sur trottoir'
 'Sur accotement' '9' 'Sur refuge ou BAU']

 Valeu

In [ ]:
# FONCTION 7 - TRANSFORMATION DF USAGERS

def clean_usager_df(df):
    df = df.copy()

    # 1️⃣ Normalize case and strip whitespace (avoids ' nan' or 'None ')
    df = df.apply(lambda col: col.astype(str).str.strip() if col.dtype == "object" else col)

    # 2️⃣ Replace 'nan', 'NaN', 'None', '-1' (string forms) with real np.nan
    df.replace(to_replace=['nan', 'NaN', 'None', '-1'], value=np.nan, inplace=True)

    # 3️⃣ Replace actual Python None or np.nan are already covered by the above
    # (no need to fillna, pandas already handles that)

    # 4️⃣ For 'locp' and 'actp' — digits, 'A', 'B' → np.nan
    df['locp'] = df['locp'].replace(r'^\d+$', np.nan, regex=True)
    df['actp'] = df['actp'].replace(r'^\d+$', np.nan, regex=True)
    df['actp'] = df['actp'].replace(['A', 'B'], np.nan)

    # 5️⃣ For accidents without pedestrians, nullify pedestrian-only columns
    mask_pieton_present = (
        df.groupby("num_acc")["catu"]
        .transform(lambda x: (x == "Piéton").any())
    )
    cols_to_null = ["locp", "actp", "etatp"]
    df.loc[~mask_pieton_present, cols_to_null] = np.nan

    return df


In [ ]:
# EXPLORATION

# Test clean_usager_df fonction
df_usager_clean = clean_usager_df(df_usager_exploded)
df_usager_clean.head()

,usager_id,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,1,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
1,2,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Passager,2,NaN,NaN,NaN,201900020750
2,3,Masculin,Indemne,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
3,4,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020796
4,5,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020869


In [ ]:
# EXPLORATION

df_usager_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1061254 entries, 0 to 1061253
Data columns (total 12 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   usager_id  1061254 non-null  int64 
 1   sexe       1061254 non-null  object
 2   grav       1061254 non-null  object
 3   trajet     778342 non-null   object
 4   secu       885685 non-null   object
 5   secu_utl   885685 non-null   object
 6   catu       1061254 non-null  object
 7   place      978364 non-null   object
 8   locp       84065 non-null    object
 9   actp       172739 non-null   object
 10  etatp      87841 non-null    object
 11  num_acc    1061254 non-null  int64 
dtypes: int64(2), object(10)
memory usage: 97.2+ MB


In [ ]:
# EXPLORATION

for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    print(f"\nColumn: {col}")
    print(df_usager_clean[col].value_counts(dropna=False).head())


Column: sexe
sexe
Masculin    719688
Féminin     341566
Name: count, dtype: int64

Column: grav
grav
Blessé     596739
Indemne    435945
Tué         28570
Name: count, dtype: int64

Column: trajet
trajet
Promenade – loisirs            406595
NaN                            282912
Domicile – travail             144371
Utilisation professionnelle    102249
Autre                           72909
Name: count, dtype: int64

Column: secu
secu
Ceinture              614903
Casque                194517
NaN                   175569
Autre                  58148
Dispositif enfants     15014
Name: count, dtype: int64

Column: secu_utl
secu_utl
Oui                 711856
NaN                 175569
Non déterminable    142429
Non                  31400
Name: count, dtype: int64

Column: catu
catu
Conducteur                            788583
Passager                              178561
Piéton                                 92477
Piéton en roller ou en trottinette      1633
Name: count, dtype: int64

Co

In [ ]:
# EXPLORATION

# Controller les valeurs uniques des catégories après nettoyage
for col in ['sexe', 'grav', 'trajet', 'secu', 'secu_utl', 'catu', 'place', 'locp', 'actp', 'etatp']:
    unique_values = df_usager_clean[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")  


 Valeurs uniques dans 'sexe': ['Masculin' 'Féminin']

 Valeurs uniques dans 'grav': ['Blessé' 'Indemne' 'Tué']

 Valeurs uniques dans 'trajet': ['Promenade – loisirs' nan 'Utilisation professionnelle'
 'Domicile – travail' 'Courses – achats' 'Autre' 'Domicile – école']

 Valeurs uniques dans 'secu': [nan 'Ceinture' 'Casque' 'Autre' 'Equipement réfléchissant'
 'Dispositif enfants']

 Valeurs uniques dans 'secu_utl': [nan 'Oui' 'Non déterminable' 'Non']

 Valeurs uniques dans 'catu': ['Conducteur' 'Passager' 'Piéton' 'Piéton en roller ou en trottinette']

 Valeurs uniques dans 'place': ['1' '2' '9' '7' '3' '4' nan '5' '10' '8' '6']

 Valeurs uniques dans 'locp': [nan 'Sur contre allée'
 'Sur passage piéton - Avec signalisation lumineuse'
 'Sur passage piéton - Sans signalisation lumineuse'
 'Sur chaussée - A – 50 m du passage piéton'
 'Sur chaussée - A + 50 m du passage piéton' 'Sur trottoir'
 'Sur accotement' 'Sur refuge ou BAU']

 Valeurs uniques dans 'actp': [nan 'Se déplaçant' 'Trav

In [71]:
# Dataframe usagers prêt à l'export

df_usager_clean.head(10)

,usager_id,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,1,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
1,2,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Passager,2,NaN,NaN,NaN,201900020750
2,3,Masculin,Indemne,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020750
3,4,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020796
4,5,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020869
5,6,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Passager,9,NaN,NaN,NaN,201900020869
6,7,Masculin,Blessé,Promenade – loisirs,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900020869
7,8,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Passager,7,NaN,NaN,NaN,201900020869
8,9,Masculin,Indemne,Promenade – loisirs,NaN,NaN,Passager,2,NaN,NaN,NaN,201900020869
9,10,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,NaN,NaN,NaN,201900021309


# ZOUBIR